In [102]:
!pip install pytorch-crf

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [103]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AdamW, AutoTokenizer, AutoModelForTokenClassification, AutoConfig
from torchcrf import CRF 
from torch.utils.data import DataLoader
from tqdm import tqdm

In [104]:

from datasets import load_dataset, load_from_disk
#from transformers import Trainer, TrainingArguments
from seqeval.metrics import accuracy_score, f1_score, precision_score, recall_score

In [105]:
from datasets import load_from_disk

In [106]:
ROOT_DIR="/Users/pals/MICS/MIDS_266/project/privacy-ner-att"
dataset_name=f"{ROOT_DIR}/datasets/hf_expanded_privacy_dataset"

In [107]:
dataset = load_from_disk(dataset_name)
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'text', 'annotated_text', 'privacy_class_label'],
        num_rows: 3000
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'text', 'annotated_text', 'privacy_class_label'],
        num_rows: 600
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'text', 'annotated_text', 'privacy_class_label'],
        num_rows: 400
    })
})


In [108]:
model_name = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [113]:
count=0
for row in dataset["train"]:
    print(row["ner_tags"])
    count += 1
    if count >=1:
        break


['Ojjn', 'Bpkjcg', 'was', 'admitted', 'to', 'Hilltop', 'Clinic', 'for', 'assault', 'charge', '.', 'They', 'received', 'treatment', 'arrest', 'and', 'are', 'currently', 'recovering', '.', 'Their', 'SSN', 'social security number', 'and', 'credit', 'card', '5500000000000004', 'were', 'recorded', 'at', 'admission', '.', 'Official', 'contact:', 'ojjn_bpkjcg@email', '.net', '.', 'The', 'hospital', 'ensured', 'patient', 'privacy', 'during', 'treatment', '.']


In [112]:
for tag,token in row["bio_tags"]:
    print(tag,token)


ValueError: too many values to unpack (expected 2)

In [114]:
# Extract full tag sets from training split
def extract_tags(dataset):
    all_ner_tags = set(["O"])
    all_pii_tags = set(["O"])

    for row in dataset["train"]:
        try:
            bio_tags = eval(row["bio_tags"])
        except:
            continue  # skip malformed rows

        for tag, token in bio_tags:
            if tag == "O":
                continue
            entity_type = tag.split("-")[-1]
            if entity_type == "PER":
                all_ner_tags.add(tag)
            else:
                all_pii_tags.add(tag)

    return sorted(all_ner_tags), sorted(all_pii_tags)


In [115]:
ner_labels, pii_labels = extract_tags(dataset)
personal_label2id = {tag: i for i, tag in enumerate(ner_labels)}
pii_label2id = {tag: i for i, tag in enumerate(pii_labels)}
personal_labels = [v for v in personal_label2id.values()]
pii_labels = [v for v in pii_label2id.values()]

print("Personal Labels to ID:", personal_label2id)
print("PII Labels to ID:", pii_label2id)
print("personal_labels:", personal_labels)
print("pii_labels:", pii_labels)

Personal Labels to ID: {'O': 0}
PII Labels to ID: {'O': 0}
personal_labels: [0]
pii_labels: [0]


In [12]:
# Tokenize and align
def tokenize_and_align_labels(input_txt):
    tokens = input_txt["text"].replace(".", " .").replace(",", " ,").split()
    bio_tags = eval(input_txt["bio_tags"])
    ner_labels, pii_labels = [], []

    for tag, _ in bio_tags:
        ner_labels.append(ner_tag2id.get(tag, 0) if tag.startswith("B-PER") or tag.startswith("I-PER") else ner_tag2id["O"])
        pii_labels.append(pii_tag2id.get(tag, pii_tag2id["O"]) if tag in pii_tag2id else pii_tag2id["O"])

    tokenized = tokenizer(tokens, truncation=True, padding="max_length", max_length=256, is_split_into_words=True)
    word_ids = tokenized.word_ids()
    tokenized["labels_ner"] = [ner_labels[w] if w is not None else -100 for w in word_ids]
    tokenized["labels_pii"] = [pii_labels[w] if w is not None else -100 for w in word_ids]
    return tokenized

tokenized_dataset = dataset.map(tokenize_and_align_labels)

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

In [28]:
print(tokenized_dataset["train"].features)

{'text': Value(dtype='string', id=None), 'bio_tags': Value(dtype='string', id=None), 'annotated_text': Value(dtype='string', id=None), 'input_ids': Sequence(feature=Value(dtype='int32', id=None), length=-1, id=None), 'attention_mask': Sequence(feature=Value(dtype='int8', id=None), length=-1, id=None), 'labels_ner': Sequence(feature=Value(dtype='int64', id=None), length=-1, id=None), 'labels_pii': Sequence(feature=Value(dtype='int64', id=None), length=-1, id=None)}


In [116]:
class CRFNERHead(nn.Module):
    """
    NER classifier with optional expandable feedforward network before CRF.
    """
    def __init__(self, hidden_size, label2id, expansion_factor=1):
        super(CRFNERHead, self).__init__()
        self.label2id = label2id
        self.id2label = {v: k for k, v in label2id.items()}
        self.num_labels = len(label2id)
        self.expanded_size = int(hidden_size * expansion_factor)

        self.ff_layer = nn.Linear(hidden_size, self.expanded_size)
        self.activation = nn.ReLU()
        self.classifier = nn.Linear(self.expanded_size, self.num_labels)
        self.crf = CRF(num_tags=self.num_labels, batch_first=True)

    def forward(self, sequence_output, labels=None, attention_mask=None, return_decoded=False):
        """
        Forward pass:
        - sequence_output: (batch, seq_len, hidden_size)
        - labels: Ground truth token labels (batch, seq_len)
        - attention_mask: Padding mask (batch, seq_len)
        - return_decoded: If True, returns decoded tag strings

        Returns:
        - logits, loss (optional), decoded tags (optional)
        """
        x = self.ff_layer(sequence_output)
        x = self.activation(x)
        logits = self.classifier(x)

        mask = attention_mask.bool() if attention_mask is not None else None

        loss = None
        decoded_labels = None

        if labels is not None:
            loss = -self.crf(logits, labels, mask=mask, reduction='mean')
        if return_decoded:
            decoded_ids = self.crf.decode(logits, mask=mask)
            decoded_labels = [
                [self.id2label[tag_id] for tag_id in seq] for seq in decoded_ids
            ]

        return logits, loss, decoded_labels


In [117]:
class CrossAttentionLayer(nn.Module):
    def __init__(self, hidden_size, personal_label2id, pii_label2id):
        """
        Cross-attention between personal NER tokens and PII tokens.

        Args:
        - hidden_size: Hidden size of transformer output
        - personal_label2id: Dict for personal entity tags (e.g., B-PER, I-PER)
        - pii_label2id: Dict for privacy-sensitive tags (e.g., B-SSN, B-CONDITION)
        """
        super().__init__()
        self.query_dense = nn.Linear(hidden_size, hidden_size)
        self.key_dense = nn.Linear(hidden_size, hidden_size)
        self.value_dense = nn.Linear(hidden_size, hidden_size)

        # Derive list of relevant label IDs from input dicts
        self.personal_labels = [personal_label2id[lbl] for lbl in personal_label2id if lbl != "O"]
        self.pii_labels = [pii_label2id[lbl] for lbl in pii_label2id if lbl != "O"]

    def forward(self,
                sequence_output,               # [B, T, H]
                logits_ner=None,               # [B, T, C1]
                logits_pii=None,               # [B, T, C2]
                labels_ner=None,               # [B, T]
                labels_pii=None,               # [B, T]
                attention_mask=None):          # [B, T]

        batch_size, seq_len, hidden_dim = sequence_output.size()

        # === TRAINING MODE ===
        if labels_ner is not None and labels_pii is not None:
            personal_mask = torch.zeros_like(labels_ner, dtype=torch.float)
            pii_mask = torch.zeros_like(labels_pii, dtype=torch.float)

            for tag_id in self.personal_labels:
                personal_mask += (labels_ner == tag_id).float()
            for tag_id in self.pii_labels:
                pii_mask += (labels_pii == tag_id).float()

            personal_repr = sequence_output * personal_mask.unsqueeze(-1)
            pii_repr = sequence_output * pii_mask.unsqueeze(-1)

        # === INFERENCE MODE ===
        elif logits_ner is not None and logits_pii is not None:
            ner_probs = F.softmax(logits_ner, dim=-1)
            pii_probs = F.softmax(logits_pii, dim=-1)

            personal_probs = ner_probs[..., self.personal_labels].sum(dim=-1, keepdim=True)
            pii_probs = pii_probs[..., self.pii_labels].sum(dim=-1, keepdim=True)

            personal_repr = sequence_output * personal_probs
            pii_repr = sequence_output * pii_probs

        else:
            raise ValueError("Must provide either (labels_ner & labels_pii) or (logits_ner & logits_pii)")

        # === QKV Attention ===
        Q = self.query_dense(personal_repr)
        K = self.key_dense(pii_repr)
        V = self.value_dense(pii_repr)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / (hidden_dim ** 0.5)  # [B, T, T]

        if attention_mask is not None:
            scores = scores.masked_fill(attention_mask.unsqueeze(1) == 0, float("-inf"))

        attn_weights = F.softmax(scores, dim=-1)
        attended_output = torch.matmul(attn_weights, V)

        return attended_output, attn_weights


In [118]:
class PrivacyClassificationHead(nn.Module):
    def __init__(self, hidden_size, num_classes=2, use_max_pool=True, expansion_factor=1.0):
        super().__init__()
        self.use_max_pool = use_max_pool
        self.num_classes = num_classes

        intermediate_size = int(hidden_size * expansion_factor)
        self.ff = nn.Linear(hidden_size, intermediate_size)
        self.activation = nn.ReLU()
        self.classifier = nn.Linear(intermediate_size, num_classes)
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, hidden_states, attention_mask, labels=None, return_attention=False, attention_weights=None):
        """
        hidden_states: [B, T, H]
        attention_mask: [B, T]
        labels: [B] → 0/1 for binary classification
        """
        if self.use_max_pool:
            masked = hidden_states.masked_fill(attention_mask.unsqueeze(-1) == 0, -1e9)
            pooled = torch.max(masked, dim=1).values  # [B, H]
        else:
            masked = hidden_states * attention_mask.unsqueeze(-1)
            pooled = torch.sum(masked, dim=1) / attention_mask.sum(dim=1, keepdim=True)  # [B, H]

        x = self.ff(pooled)  # [B, H']
        x = self.activation(x)
        logits = self.classifier(x)  # [B, num_classes]

        if labels is not None:
            loss = self.loss_fn(logits, labels)
            return logits, loss

        return logits, attention_weights if return_attention else logits


In [123]:
class PrivacyDetectionModel(nn.Module):
    def __init__(
        self,
        base_model_name: str,
        personal_label2id: dict,
        pii_label2id: dict,
        num_privacy_labels: int = 2,
        return_attention_weights: bool = False,
        expansion_factor: float = 1.0
    ):
        super().__init__()

        self.base_model = AutoModel.from_pretrained(base_model_name)
        hidden_size = self.base_model.config.hidden_size
        self.return_attention_weights = return_attention_weights

        # === NER Heads ===

        self.personal_ner_head = CRFNERHead(
            hidden_size=hidden_size,
            label2id=personal_label2id,
            expansion_factor=expansion_factor
        )

        self.pii_ner_head = CRFNERHead(
            hidden_size=hidden_size,
            label2id=pii_label2id,
            expansion_factor=expansion_factor
        )

        # === Cross Attention Layer ===
        self.cross_attention = CrossAttentionLayer(
            hidden_size=hidden_size,
            personal_label2id=personal_label2id,
            pii_label2id=pii_label2id
        )

        # === Final Privacy Classification Head ===
        self.classifier_head = PrivacyClassificationHead(
            hidden_size=hidden_size,
            num_classes=num_privacy_labels,
            use_max_pool=True,
            expansion_factor=expansion_factor
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        labels_ner=None,
        labels_pii=None,
        privacy_labels=None,
        return_decoded=False,
    ):
        """
        Args:
            input_ids: [B, T]
            attention_mask: [B, T]
            labels_ner: Optional [B, T]
            labels_pii: Optional [B, T]
            privacy_labels: Optional [B]
            return_decoded: Optional bool to return predicted tag strings

        Returns:
            Dict with keys: loss, logits, decoded_ner, decoded_pii, attention_weights
        """

        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state  # [B, T, H]

        # === NER Predictions ===
        logits_ner, loss_ner, decoded_ner = self.personal_ner_head(
            sequence_output, labels=labels_ner, attention_mask=attention_mask, return_decoded=return_decoded
        )

        logits_pii, loss_pii, decoded_pii = self.pii_ner_head(
            sequence_output, labels=labels_pii, attention_mask=attention_mask, return_decoded=return_decoded
        )

        # === Cross Attention between personal + pii tokens ===
        attended_output, attention_weights = self.cross_attention(
            sequence_output=sequence_output,
            logits_ner=logits_ner if labels_ner is None else None,
            logits_pii=logits_pii if labels_pii is None else None,
            labels_ner=labels_ner,
            labels_pii=labels_pii,
            attention_mask=attention_mask
        )

        # === Final Privacy Classification ===
        logits_privacy, loss_privacy = self.classifier_head(
            attended_output, labels=privacy_labels,
            attention_mask=attention_mask,
            return_attention_weights=self.return_attention_weights
        )

        # === Total Loss (NER + Privacy)
        total_loss = 0
        if loss_ner is not None:
            total_loss += loss_ner
        if loss_pii is not None:
            total_loss += loss_pii
        if loss_privacy is not None:
            total_loss += loss_privacy

        result = {
            "loss": total_loss,
            "logits_privacy": logits_privacy,
        }

        if return_decoded:
            result["decoded_ner"] = decoded_ner
            result["decoded_pii"] = decoded_pii

        if self.return_attention_weights:
            result["attention_weights"] = attention_weights

        return result


In [124]:
base_model_name = "answerdotai/ModernBERT-base"
privacy_model = PrivacyDetectionModel(base_model_name, personal_label2id=personal_label2id, pii_label2id=pii_label2id)
print(privacy_model)

PrivacyDetectionModel(
  (base_model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50368, 768, padding_idx=50283)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=2304, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=1152, out_features=768, bias=False)
        )
      )
      (

In [58]:
device = torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
print(f"Loading model to device: {device}")
privacy_model.to(device)

Loading model to device: mps


PrivacyDetectionModel(
  (base_model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50368, 768, padding_idx=50283)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=2304, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=1152, out_features=768, bias=False)
        )
      )
      (

In [59]:


def compute_metrics(pred):
    logits, labels = pred

    # Convert logits to predicted class indices
    predictions = np.argmax(logits, axis=2)

    true_labels = []
    pred_labels = []

    for label_row, pred_row in zip(labels, predictions):  # Process each sentence
        temp_true = []
        temp_pred = []
        
        for label_id, pred_id in zip(label_row, pred_row):  # Process each token
            if label_id != -100:  # Ignore padding tokens
                temp_true.append(ner_id2tag[label_id])  # Convert true label to string
                temp_pred.append(ner_id2tag[pred_id])  # Convert predicted label to string

        if temp_true:  # Only add non-empty sequences
            true_labels.append(temp_true)
            pred_labels.append(temp_pred)

    return {
        "accuracy": accuracy_score(true_labels, pred_labels),
        "f1": f1_score(true_labels, pred_labels),
        "precision": precision_score(true_labels, pred_labels),
        "recall": recall_score(true_labels, pred_labels),
    }


In [83]:
def custom_collate_fn(batch):
    return {
        "input_ids": torch.tensor([item["input_ids"] for item in batch], dtype=torch.long),
        "attention_mask": torch.tensor([item["attention_mask"] for item in batch], dtype=torch.long),
        "labels_ner": torch.tensor([item["labels_ner"] for item in batch], dtype=torch.long),
        "labels_pii": torch.tensor([item["labels_pii"] for item in batch], dtype=torch.long),
    }


In [84]:
MODEL   = "privacy_model"
OUTDIR  = f"{ROOT_DIR}/build/{MODEL}/results"
LOGDIR  = f"{ROOT_DIR}/build/{MODEL}/logs"
SAVEDIR = f"{ROOT_DIR}/build/{MODEL}/checkpoints"
BATCH_SIZE = 8
NUM_EPOCHS = 10
WT_DECAY = 0.01
LEARNING_RATE = 2E-5

In [85]:
def train_privacy_model(model,
                        dataset,
                        tokenizer,
                        num_epochs=5,
                        learning_rate=2e-5,
                        batch_size=8,
                        save_path=None,
                        save_every_epoch=True):
    # Auto-detect device
    device = (
        torch.device("mps") if torch.backends.mps.is_available()
        else torch.device("cuda") if torch.cuda.is_available()
        else torch.device("cpu")
    )
    print(f"🔧 Using device: {device}")

    model.to(device)
    model.train()

    # Prepare DataLoaders
    train_loader = DataLoader(dataset["train"], batch_size=batch_size, shuffle=True, collate_fn=custom_collate_fn)
    val_loader = DataLoader(dataset["validation"], batch_size=batch_size, collate_fn=custom_collate_fn)

    optimizer = AdamW(model.parameters(), lr=learning_rate)

    for epoch in range(num_epochs):
        print(f"\n Epoch {epoch + 1}/{num_epochs}")
        total_train_loss = 0
        total_val_loss = 0

        # === Training ===
        model.train()
        for batch in tqdm(train_loader, desc="Training"):
            input_ids = torch.tensor(batch["input_ids"]).to(device)
            attention_mask = torch.tensor(batch["attention_mask"]).to(device)
            labels_ner = torch.tensor(batch["labels_ner"]).to(device)
            labels_pii = torch.tensor(batch["labels_pii"]).to(device)

            outputs = model(input_ids=input_ids,
                            attention_mask=attention_mask,
                            labels_ner=labels_ner,
                            labels_pii=labels_pii)

            loss = outputs["loss"]
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            total_train_loss += loss.item()

        avg_train_loss = total_train_loss / len(train_loader)
        print(f"Training Loss: {avg_train_loss:.4f}")

        # === Validation ===
        model.eval()
        with torch.no_grad():
            for batch in tqdm(val_loader, desc="Validation"):
                input_ids = torch.tensor(batch["input_ids"]).to(device)
                attention_mask = torch.tensor(batch["attention_mask"]).to(device)
                labels_ner = torch.tensor(batch["labels_ner"]).to(device)
                labels_pii = torch.tensor(batch["labels_pii"]).to(device)

                outputs = model(input_ids=input_ids,
                                attention_mask=attention_mask,
                                labels_ner=labels_ner,
                                labels_pii=labels_pii)
                loss = outputs["loss"]
                total_val_loss += loss.item()

        avg_val_loss = total_val_loss / len(val_loader)
        print(f"Validation Loss: {avg_val_loss:.4f}")

        # === Save Model Checkpoint ===
        if save_path and save_every_epoch:
            os.makedirs(save_path, exist_ok=True)
            model_save_file = os.path.join(save_path, f"privacy_model_epoch{epoch + 1}.pt")
            torch.save(model.state_dict(), model_save_file)
            print(f"Saved checkpoint to: {model_save_file}")

    print("Training complete.")
    return model


In [86]:
train_privacy_model(model = privacy_model,
                    dataset = tokenized_dataset,
                    tokenizer = tokenizer,
                    num_epochs=NUM_EPOCHS,
                    learning_rate=LEARNING_RATE,
                    batch_size=BATCH_SIZE,
                    save_path=SAVEDIR,
                    save_every_epoch=True)

🔧 Using device: mps

 Epoch 1/10


/var/folders/nx/k40f40dj4ddcngbqkb3jvy_w0000gn/T/ipykernel_88425/1533869298.py:34: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_ids = torch.tensor(batch["input_ids"]).to(device)
/var/folders/nx/k40f40dj4ddcngbqkb3jvy_w0000gn/T/ipykernel_88425/1533869298.py:35: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  attention_mask = torch.tensor(batch["attention_mask"]).to(device)
/var/folders/nx/k40f40dj4ddcngbqkb3jvy_w0000gn/T/ipykernel_88425/1533869298.py:36: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels_ner = torch.tensor(batch["la

TypeError: PrivacyDetectionModel.forward() got an unexpected keyword argument 'labels_ner'

In [79]:
print(tokenized_dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'bio_tags', 'annotated_text', 'input_ids', 'attention_mask', 'labels_ner', 'labels_pii'],
        num_rows: 3000
    })
    validation: Dataset({
        features: ['text', 'bio_tags', 'annotated_text', 'input_ids', 'attention_mask', 'labels_ner', 'labels_pii'],
        num_rows: 600
    })
    test: Dataset({
        features: ['text', 'bio_tags', 'annotated_text', 'input_ids', 'attention_mask', 'labels_ner', 'labels_pii'],
        num_rows: 400
    })
})
